# AnchorDraw (Semantic Anchor) SD1.5 + LCM Full1073 Metric Run trên Kaggle

## Thông Tin Nhận Diện Run

```text
Method    : AnchorDraw (Semantic Anchor)
Config    : WM-03 (Adaptive Bilateral) + Layer L04 + sigma_D = 4.0 + Bootstrap = 1
Model     : SD 1.5 base
Checkpoint: runwayml/stable-diffusion-v1-5
Sampler   : LCMScheduler (5 timesteps: [999, 919, 759, 499, 259])
Accel     : latent-consistency/lcm-lora-sdv1-5/pytorch_lora_weights.safetensors
Resolution: 512x512
Manifest  : Ours/data_manifests/coco_val2017_multidiffusion_coco_all_512x512_all.jsonl
Samples   : 1073
Metrics   : FID, IS, CLIP(fg), CLIP(bg), Time(s)
Base Seed : 2024 (seed = BASE_SEED + global_index, đồng bộ bit-exact với Baseline)
Batch Size: 8 (DataLoader batch_size = 8, đồng bộ bit-exact với Baseline)
```

Notebook này chạy end-to-end phương pháp mới **AnchorDraw (Semantic Anchor)** với full manifest COCO val2017 hợp lệ theo protocol SD1.5 512x512:

`Ours/data_manifests/coco_val2017_multidiffusion_coco_all_512x512_all.jsonl`

Full manifest có **1073 sample**. Mỗi sample tương ứng một ảnh COCO hợp lệ sau bước filter của dataloader/manifest.

**Các thông số và điều kiện thực nghiệm được đồng bộ 100% với baseline SemanticDraw** (`semanticdraw_sd15_lcm_full1073_kaggle.ipynb`):
- `BASE_SEED = 2024` (từng ảnh sinh với seed giống hệt baseline).
- `BATCH_SIZE = 8` (dataloader batching giống hệt baseline).
- `TARGET_SIZE = (512, 512)`, `BOOTSTRAP_STEPS = 1`, `MASK_STD = 0.0`, `MASK_STRENGTH = 1.0`, `PREPROCESS_MASK_COVER_ALPHA = 0.0`.

**Cốt lõi cải tiến của AnchorDraw trong run này:**
1. **Dynamic Centering via Semantic Anchor:** Thay vì chỉ căn giữa hình học ở bootstrap, các bước sau neo động vật thể qua điểm kích hoạt Cross-Attention (`semantic_topk_anchor`, Top-10% attention centroid).
2. **Layer cố định:** **L04** (`down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor`).
3. **Chính sách trọng số Mask:** **WM-03 (`adaptive_bilateral`)** với bộ lọc song phương không gian ($\sigma_D = 4.0$) và ngữ nghĩa thích ứng ($\sigma_S$), kết hợp dải biên hình thái học và background absorption để đảm bảo Partition of Unity $\sum W_k = 1.0$.

Sau khi sinh ảnh xong, notebook tự động chạy phần đo metric:
`FID`, `IS`, `CLIP(fg)`, `CLIP(bg)`, `Time(s)`

Kết quả metric được lưu tại:
`/kaggle/working/semantic_anchor_sd15_lcm_w03_l04_sigmad4_full1073_metrics/`


## 0. Yêu cầu Kaggle

- Bật GPU trong `Settings -> Accelerator -> GPU`.
- Bật Internet để tải COCO, Stable Diffusion 1.5, LCM LoRA, Inception weights và CLIP weights.
- Nếu Hugging Face yêu cầu quyền truy cập, thêm Kaggle Secret tên `HF_TOKEN` hoặc set biến môi trường `HF_TOKEN`.
- Khi chạy full, giữ `MAX_DISPLAY_RESULTS = 8` để notebook không phình RAM vì hiển thị quá nhiều ảnh.


In [ ]:
# Cài các thư viện cần thiết cho generation + metric evaluation.
# Không cài lại torch để tránh làm lệch môi trường GPU mặc định của Kaggle.
# Quan trọng: gỡ torchao. Một số Kaggle image có torchao==0.10.0;
# peft mới thấy torchao nhưng yêu cầu >0.16.0, gây lỗi khi load LCM LoRA.
import sys
import subprocess

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate",
    "peft",
    "huggingface_hub",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "einops",
    "pycocotools",
    "torchmetrics",
    "clean-fid",
    "torch-fidelity",
    "open-clip-torch>=2.24.0",
]

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("[OK] Dependencies installed successfully.")


In [ ]:
# Clone repo nếu notebook chưa nằm trong repo clone.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/GOx9-P/AnchorDraw.git"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

def is_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "Ours").exists()
        and (path / "Baseline").exists()
    )

def find_repo_root() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        WORK_DIR / "AnchorDraw",
        Path("/kaggle/working/AnchorDraw"),
    ]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    for root, dirs, _ in os.walk(str(WORK_DIR)):
        for d in dirs:
            path = Path(root) / d
            if is_repo_root(path):
                return path
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    clone_target = WORK_DIR / "AnchorDraw"
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    REPO_ROOT = find_repo_root()

assert REPO_ROOT is not None and is_repo_root(REPO_ROOT), "Không tìm thấy repo root sau khi clone."
print(f"[OK] Repo root: {REPO_ROOT}")


In [ ]:
# Cấu hình full manifest 1073 cho benchmark AnchorDraw SD1.5 + LCM + L04 + WM-03 + sigma_D=4.0.
from pathlib import Path

RUN_MANIFEST = REPO_ROOT / "Ours" / "data_manifests" / "coco_val2017_multidiffusion_coco_all_512x512_all.jsonl"
COCO_ROOT = Path(os.environ.get("COCO_ROOT", "/kaggle/working/COCO"))
OUTPUT_DIR = Path("/kaggle/working/semantic_anchor_sd15_lcm_w03_l04_sigmad4_full1073_outputs")
MASK_CACHE_DIR = Path("/kaggle/working/semantic_anchor_sd15_mask_cache")
METRICS_OUTPUT_DIR = Path("/kaggle/working/semantic_anchor_sd15_lcm_w03_l04_sigmad4_full1073_metrics")

# Model theo baseline SD1.5.
MODEL_ID = "runwayml/stable-diffusion-v1-5"

# --- THÔNG SỐ ĐỒNG BỘ 100% VỚI BASELINE ---
TARGET_SIZE = (512, 512)
BATCH_SIZE = 8
BASE_SEED = 2024
BOOTSTRAP_STEPS = 1
MASK_STD = 1.0  # Chuan sweep: 1.0 de Gaussian lowpass tao gradient lam mo
MASK_STRENGTH = 1.0
PREPROCESS_MASK_COVER_ALPHA = 0.0
MASK_TYPE = "discrete"
NEGATIVE_PROMPT = ""

# --- CẤU HÌNH PHƯƠNG PHÁP MỚI ANCHORDRAW ---
METHOD_ID = "WM-03-TOPK-L04-SIGMA4"
METHOD_LABEL = "AnchorDraw (Top-k + Adaptive Bilateral, L04, sigmaD=4.0)"
BEST_ANCHOR_MODE = "semantic_topk_anchor"
TOPK_ATTENTION_PERCENT = 10.0
WEIGHT_POLICY = "adaptive_bilateral"
FIXED_LAYER = {
    "layer_id": "L04",
    "layer_name": "down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor",  # L04 chuan (16x16)
}
SIGMA_D = 4.0
SEMANTIC_SIGMA_SCALE = 1.0

# Chỉ hiển thị 8 preview để Kaggle notebook không bị nặng.
# Ảnh vẫn được sinh và lưu đủ theo full manifest 1073.
MAX_DISPLAY_RESULTS = 8

# Cấu hình đo Metric (Đồng bộ 100% với baseline)
METRIC_NAMES = ("fid", "is", "clip_fg", "clip_bg", "time")
METRIC_BATCH_SIZE = 8
CLIP_BATCH_SIZE = 16
IS_SPLITS = 10
METRICS_REPORT_PREFIX = "semantic_anchor_sd15_lcm_w03_l04_sigmad4_full1073_metrics"

assert RUN_MANIFEST.exists(), f"Missing manifest: {RUN_MANIFEST}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Manifest: {RUN_MANIFEST}")
print(f"[OK] COCO root: {COCO_ROOT}")
print(f"[OK] Output dir: {OUTPUT_DIR}")
print(f"[OK] Metrics output dir: {METRICS_OUTPUT_DIR}")
print(f"[OK] Batch size: {BATCH_SIZE} (Sync với Baseline)")
print(f"[OK] Base seed: {BASE_SEED} (Sync với Baseline)")
print(f"[OK] Method: {METHOD_ID} | Layer: {FIXED_LAYER['layer_id']} | sigma_D: {SIGMA_D}")


In [ ]:
# Tải COCO val2017 nếu Kaggle runtime chưa có sẵn dữ liệu.
import ssl
import urllib.request
import zipfile

COCO_ROOT.mkdir(parents=True, exist_ok=True)

VAL_ZIP_URLS = [
    "http://images.cocodataset.org/zips/val2017.zip",
    "https://images.cocodataset.org/zips/val2017.zip",
]
ANN_ZIP_URLS = [
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
]
val_zip = COCO_ROOT / "val2017.zip"
ann_zip = COCO_ROOT / "annotations_trainval2017.zip"

def run_download_command(cmd: list[str]) -> bool:
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as exc:
        print(f"[WARN] Download command failed: {' '.join(cmd[:2])} -> {exc}")
        return False

def download_file(urls: list[str], dst: Path) -> None:
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[SKIP] Already downloaded: {dst.name}")
        return

    last_error = None
    for url in urls:
        print(f"[DOWNLOAD] Trying: {url}")
        if run_download_command(["wget", "-c", "--no-check-certificate", "-O", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                print(f"[OK] Downloaded with wget: {dst.name}")
                return
        if run_download_command(["curl", "-k", "-L", "-C", "-", "-o", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                print(f"[OK] Downloaded with curl: {dst.name}")
                return
        try:
            ctx = ssl.create_default_context()
            ctx.check_hostname = False
            ctx.verify_mode = ssl.CERT_NONE
            with urllib.request.urlopen(url, context=ctx) as response, dst.open("wb") as f:
                f.write(response.read())
            if dst.exists() and dst.stat().st_size > 0:
                print(f"[OK] Downloaded with urllib: {dst.name}")
                return
        except Exception as exc:
            last_error = exc
            print(f"[WARN] urllib failed for {url}: {exc}")

    raise RuntimeError(f"Không thể tải {dst.name} từ các URL đã thử. Lỗi cuối: {last_error}")

def unzip_if_missing(zip_path: Path, probe_file: Path) -> None:
    if probe_file.exists():
        print(f"[SKIP] Already unzipped: {probe_file}")
        return
    print(f"[UNZIP] Unpacking {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(COCO_ROOT)
    print(f"[OK] Unpacked {zip_path.name}")

download_file(VAL_ZIP_URLS, val_zip)
download_file(ANN_ZIP_URLS, ann_zip)
unzip_if_missing(val_zip, COCO_ROOT / "val2017" / "000000000139.jpg")
unzip_if_missing(ann_zip, COCO_ROOT / "annotations" / "instances_val2017.json")

assert (COCO_ROOT / "val2017").exists(), "Missing COCO val2017 images."
assert (COCO_ROOT / "annotations" / "instances_val2017.json").exists(), "Missing instances_val2017.json."
assert (COCO_ROOT / "annotations" / "captions_val2017.json").exists(), "Missing captions_val2017.json."
print("[OK] COCO val2017 is ready.")


In [ ]:
# Import dataloader của Ours, SemanticDrawPipeline và core AnchorDraw modules.
import sys
import importlib.util
import json
import time
import types

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

OURS_SRC = REPO_ROOT / "Ours" / "src"
BASELINE_SRC = REPO_ROOT / "Baseline" / "semantic-draw-main" / "src"

sys.path.insert(0, str(OURS_SRC))
from data import COCORegionConfig, build_coco_region_dataloader, batch_item_to_semanticdraw_inputs
from data.visualize import make_mask_overlay
from experiments.semantic_anchor import (
    SemanticAnchorCapture,
    SemanticAnchorRuntime,
    find_target_token_indices,
    compute_anchor_measurements,
    aggregate_attention_maps,
)

# Load trực tiếp file pipeline_semantic_draw.py để tránh import toàn bộ model/__init__.py
# vì các file SDXL/SD3 có thể cần dependency khác không dùng trong SD1.5.
sys.path.insert(0, str(BASELINE_SRC))
pipeline_path = BASELINE_SRC / "model" / "pipeline_semantic_draw.py"
spec = importlib.util.spec_from_file_location("pipeline_semantic_draw", pipeline_path)
pipeline_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(pipeline_module)
SemanticDrawPipeline = pipeline_module.SemanticDrawPipeline

print("[OK] All imports (DataLoader + Pipeline + AnchorDraw core) are ready.")


In [ ]:
# Tạo dataloader cho full manifest 1073 (Đồng bộ 100% cấu hình với Baseline).
config = COCORegionConfig(
    coco_root=COCO_ROOT,
    split="val2017",
    instances_json=COCO_ROOT / "annotations" / "instances_val2017.json",
    captions_json=COCO_ROOT / "annotations" / "captions_val2017.json",
    manifest_path=RUN_MANIFEST,
    profile="multidiffusion_coco_all",
    model_family="sd15",
    target_size=TARGET_SIZE,
    return_image=True,
    cache_resized_masks=True,
    cache_dir=MASK_CACHE_DIR,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)
loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
dataset_size = len(loader.dataset)
num_batches = len(loader)
preview_batch = next(iter(loader))

print(f"[OK] Manifest records: {dataset_size}")
print(f"[OK] Dataloader batches: {num_batches} batch(es) x up to {BATCH_SIZE} sample(s)")
print(f"[OK] First batch size: {len(preview_batch['sample_ids'])}")
print(f"[OK] First batch masks shape: {tuple(preview_batch['masks'].shape)}  # (B, Pmax, C, H, W)")
print("First batch sample IDs:")
for sample_id in preview_batch["sample_ids"]:
    print(" -", sample_id)


## 1. Chuẩn hóa input và Layer Selector cho AnchorDraw

Manifest lưu foreground object masks/prompts và background caption riêng.
Ở đây ta tạo:
`background_mask = 1 - union(foreground_masks)`
Input cho pipeline là:
`prompts = [COCO caption] + foreground_prompts`
`masks = [background_mask] + foreground_masks`

Ngoài ra, hàm `install_runtime_patches` gắn bộ chọn tầng attention cho layer cố định **L04**.


In [ ]:
def md_escape(text: object) -> str:
    return str(text).replace("\n", " ").replace("|", "\\|")

def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_semanticdraw_payload(batch: dict, index: int) -> dict:
    item = batch_item_to_semanticdraw_inputs(batch, index)
    fg_masks = item["masks"].float().cpu()  # (p, 1, H, W)
    fg_union = fg_masks.sum(dim=0, keepdim=True).clamp(0, 1)
    background_mask = (1.0 - fg_union).clamp(0, 1)
    all_masks = torch.cat([background_mask, fg_masks], dim=0)

    prompts = [item["background_prompt"], *item["prompts"]]
    negative_prompts = [NEGATIVE_PROMPT for _ in prompts]
    metadata = item["metadata"]

    return {
        "sample_id": item["metadata"]["sample_id"],
        "image_id": item["metadata"]["image_id"],
        "file_name": item["metadata"]["file_name"],
        "height": item["height"],
        "width": item["width"],
        "prompts": prompts,
        "negative_prompts": negative_prompts,
        "foreground_prompts": item["prompts"],
        "category_names": metadata["category_names"],
        "annotation_ids": metadata["annotation_ids"],
        "area_ratios": metadata["area_ratios"],
        "foreground_masks": fg_masks,
        "all_masks": all_masks,
        "metadata": metadata,
    }

def select_layer_attention_maps(captured_maps, output_size, layer_name):
    selected_maps = {}
    for key, layer_maps in captured_maps.items():
        matches = [layer_map for layer_map in layer_maps if layer_map.layer_name == layer_name]
        if len(matches) != 1:
            available = [layer_map.layer_name for layer_map in layer_maps]
            raise RuntimeError(f"Expected exactly one map for {layer_name} at {key}; got {len(matches)}. Available: {available}")
        values = matches[0].values.detach().float().cpu()[None, None]
        values = F.interpolate(values, size=output_size, mode="bilinear", align_corners=False)[0, 0]
        selected_maps[key] = (values - values.min()) / (values.max() - values.min()).clamp_min(1e-8)
    return selected_maps

def install_runtime_patches(runtime, target_layer_name):
    """Install Layer L04 selection with support for both argmax and topk."""
    def anchors_from_current_step(self, timestep, foreground_masks, *, strategy, topk_percent, layer_index=None, layer_name=None, **kwargs):
        eff_layer = layer_name or target_layer_name
        maps = select_layer_attention_maps(self.attention_capture.maps, self.image_size, eff_layer)
        anchors = []
        for region_index, mask in enumerate(foreground_masks):
            measurement = compute_anchor_measurements(
                maps[(int(timestep), region_index)], mask.cpu(), topk_percent=topk_percent
            )
            if strategy == "argmax":
                anchors.append((float(measurement["anchor_x"]), float(measurement["anchor_y"])))
            else:
                anchors.append((float(measurement["topk_anchor_x"]), float(measurement["topk_anchor_y"])))
        return anchors
    runtime._anchors_from_current_step = types.MethodType(anchors_from_current_step, runtime)

def display_smoke_result(payload: dict, original: Image.Image, overlay: Image.Image, generated: Image.Image, elapsed: float, generated_path: Path) -> None:
    rows = ["| Region | Prompt | Annotation | Area ratio |", "|---|---|---:|---:|"]
    rows.append(f"| Background | {md_escape(payload['prompts'][0])} | - | - |")
    for label, prompt, ann_id, area in zip(payload["category_names"], payload["foreground_prompts"], payload["annotation_ids"], payload["area_ratios"]):
        rows.append(f"| {md_escape(label)} | {md_escape(prompt)} | {ann_id} | {float(area):.4f} |")

    display(Markdown(
        f"### `{payload['sample_id']}`\n"
        f"- image_id: `{payload['image_id']}`\n"
        f"- file: `{payload['file_name']}`\n"
        f"- method: `{METHOD_ID}`\n"
        f"- generated path: `{generated_path}`\n"
        f"- elapsed: `{elapsed:.2f}s`\n\n"
        + "\n".join(rows)
    ))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original)
    axes[0].set_title("COCO original resized")
    axes[1].imshow(overlay)
    axes[1].set_title("Foreground mask overlay")
    axes[2].imshow(generated)
    axes[2].set_title(f"AnchorDraw generated ({METHOD_ID})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("[OK] Helper functions (Payload + Layer Selector + Visualization) are ready.")


In [ ]:
# Login Hugging Face nếu có token trong Kaggle Secret hoặc biến môi trường.
def maybe_login_to_huggingface() -> None:
    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import login
        login(token=token)
        print("[OK] Hugging Face token loaded.")
    else:
        print("[INFO] No HF_TOKEN found. Public/gated model access depends on your Hugging Face permissions.")

assert torch.cuda.is_available(), "Kaggle runtime chưa bật GPU. Hãy bật Accelerator = GPU rồi chạy lại."
device = torch.device("cuda:0")
dtype = torch.float16

maybe_login_to_huggingface()
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Guard chống lỗi torchao/PEFT trước khi load LCM LoRA.
import sys
import importlib
import importlib.util

importlib.invalidate_caches()

if "torchao" in sys.modules:
    raise RuntimeError(
        "torchao is already imported in this Python session. Restart the Kaggle session, run the dependency cell, then Run All."
    )

if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(
        "torchao is still installed/importable in this runtime. Run the dependency cell, then restart the Kaggle session and Run All."
    )

print("[OK] torchao is not importable; PEFT should skip torchao LoRA dispatch.")


In [ ]:
# Load SemanticDraw SD1.5 pipeline làm nền tảng cho AnchorDraw.
seed_everything(BASE_SEED)
smd = SemanticDrawPipeline(
    device=device,
    dtype=dtype,
    sd_version="1.5",
    hf_key=MODEL_ID,
    has_i2t=False,
    default_mask_std=MASK_STD,
    default_mask_strength=MASK_STRENGTH,
    default_preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
    mask_type=MASK_TYPE,
)

# Không bật attention slicing để giữ nguyên vẹn attention maps như Colab sweep

print("[OK] SemanticDrawPipeline loaded successfully.")


In [ ]:
# Kiểm tra scheduler và xác nhận layer L04 tồn tại trong UNet.
print("Scheduler:", type(smd.scheduler).__name__)
print("Default guidance scale:", smd.default_guidance_scale)
print("Default timesteps:", [int(t) for t in smd.timesteps.cpu().tolist()])

assert type(smd.scheduler).__name__ == "LCMScheduler", "Expected LCM sampler via LCMScheduler."
available_layer_names = set(smd.unet.attn_processors)
assert FIXED_LAYER["layer_name"] in available_layer_names, f"Missing {FIXED_LAYER['layer_name']} in UNet"
print(f"[OK] Fixed layer {FIXED_LAYER['layer_id']} verified: {FIXED_LAYER['layer_name']}")


In [ ]:
# Chạy generation cho mọi sample trong manifest bằng thuật toán AnchorDraw.
summary = []
global_index = 0

for batch_index, batch in enumerate(loader):
    print(f"[BATCH] {batch_index + 1}/{len(loader)} - {len(batch['sample_ids'])} sample(s)")

    for local_index, sample_id in enumerate(batch["sample_ids"]):
        payload = make_semanticdraw_payload(batch, local_index)
        original = batch["images"][local_index].resize((payload["width"], payload["height"]), Image.Resampling.BILINEAR)
        overlay = make_mask_overlay(original, payload["foreground_masks"], payload["category_names"], alpha=0.45)

        # Seed được đồng bộ bit-exact với baseline: BASE_SEED (2024) + global_index
        seed = BASE_SEED + global_index
        seed_everything(seed)

        # Xác định token vị trí cho từng đối tượng tiền cảnh
        token_indices = [
            find_target_token_indices(smd.tokenizer, prompt, category)
            for prompt, category in zip(payload["foreground_prompts"], payload["category_names"])
        ]

        tic = time.perf_counter()
        with SemanticAnchorCapture(smd.unet) as capture:
            capture.configure(token_indices)
            runtime = SemanticAnchorRuntime(smd, capture, image_size=TARGET_SIZE)
            install_runtime_patches(runtime, FIXED_LAYER["layer_name"])

            generated, runtime_records = runtime.generate(
                prompts=payload["prompts"],
                negative_prompts=payload["negative_prompts"],
                masks=payload["all_masks"].to(device=device, dtype=torch.float32),
                foreground_masks=payload["foreground_masks"],
                mode=BEST_ANCHOR_MODE,
                weight_policy=WEIGHT_POLICY,
                bootstrap_steps=BOOTSTRAP_STEPS,
                topk_percent=TOPK_ATTENTION_PERCENT,
                spatial_sigma_latent=float(SIGMA_D),
                semantic_sigma_scale=float(SEMANTIC_SIGMA_SCALE),
                mask_stds=MASK_STD,
                mask_strengths=MASK_STRENGTH,
                preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
                attention_layer_name=FIXED_LAYER["layer_name"],
            )
        elapsed = time.perf_counter() - tic

        stem = f"{global_index:04d}_{payload['sample_id']}"
        generated_path = OUTPUT_DIR / f"{stem}_generated.png"
        overlay_path = OUTPUT_DIR / f"{stem}_overlay.png"
        generated.save(generated_path)
        overlay.save(overlay_path)

        summary.append({
            "index": global_index,
            "batch_index": batch_index,
            "local_index": local_index,
            "sample_id": payload["sample_id"],
            "image_id": payload["image_id"],
            "file_name": payload["file_name"],
            "seed": seed,
            "method_id": METHOD_ID,
            "anchor_mode": BEST_ANCHOR_MODE,
            "weight_policy": WEIGHT_POLICY,
            "fixed_layer": FIXED_LAYER["layer_id"],
            "spatial_sigma_latent": float(SIGMA_D),
            "semantic_sigma_scale": float(SEMANTIC_SIGMA_SCALE),
            "bootstrap_steps": BOOTSTRAP_STEPS,
            "num_regions_including_background": len(payload["prompts"]),
            "elapsed_sec": elapsed,
            "generated_path": str(generated_path),
            "overlay_path": str(overlay_path),
        })

        should_display = MAX_DISPLAY_RESULTS is None or global_index < MAX_DISPLAY_RESULTS
        if should_display:
            display_smoke_result(payload, original, overlay, generated, elapsed, generated_path)

        global_index += 1
        torch.cuda.empty_cache()

summary_path = OUTPUT_DIR / "generation_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

display(Markdown(
    f"## Done\n"
    f"Generated `{len(summary)}` image(s) from `{dataset_size}` manifest record(s). "
    f"Summary saved to `{summary_path}`."
))
summary[:5]


In [ ]:
# Export ảnh đã sinh sang folder chuẩn để đo metric/reproduce sau này.
import csv
import shutil
from pathlib import Path

METRIC_EXPORT_EXPERIMENT_ID = "semantic_anchor_sd15_lcm_w03_l04_sigmad4_full1073"
METRIC_EXPORT_ROOT = Path("/kaggle/working/anchordraw_metric_exports")
METRIC_EXPORT_DIR = METRIC_EXPORT_ROOT / METRIC_EXPORT_EXPERIMENT_ID
METRIC_EXPORT_GENERATED_DIR = METRIC_EXPORT_DIR / "generated_images"
METRIC_EXPORT_ORIGINAL_DIR = METRIC_EXPORT_DIR / "original_images"
METRIC_EXPORT_MANIFEST_JSONL = METRIC_EXPORT_DIR / "metric_generated_manifest.jsonl"
METRIC_EXPORT_MANIFEST_CSV = METRIC_EXPORT_DIR / "metric_generated_manifest.csv"
METRIC_EXPORT_SUMMARY_JSON = METRIC_EXPORT_DIR / "export_summary.json"
METRIC_EXPORT_ZIP_PATH = METRIC_EXPORT_ROOT / f"{METRIC_EXPORT_EXPERIMENT_ID}__metric_export.zip"

COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT = False

for path in [METRIC_EXPORT_DIR, METRIC_EXPORT_GENERATED_DIR]:
    path.mkdir(parents=True, exist_ok=True)
if COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT:
    METRIC_EXPORT_ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)

def _load_generation_summary_for_metric_export():
    if "summary" in globals() and isinstance(summary, list) and len(summary) > 0:
        return summary
    candidates = []
    if "RUN_SUMMARY_PATH" in globals():
        candidates.append(Path(RUN_SUMMARY_PATH))
    if "OUTPUT_DIR" in globals():
        candidates.append(Path(OUTPUT_DIR) / "generation_summary.json")
    for candidate in candidates:
        if candidate.exists():
            with candidate.open("r", encoding="utf-8") as f:
                return json.load(f)
    raise RuntimeError("Không tìm thấy `summary` hoặc generation_summary.json để export metric.")

def _load_manifest_records_by_sample_id(manifest_path: Path) -> dict:
    records = {}
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            records[record["sample_id"]] = record
    return records

def _safe_name(text: object, max_len: int = 120) -> str:
    keep = []
    for ch in str(text):
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        else:
            keep.append("_")
    joined = "".join(keep).strip("._")
    return (joined[:max_len] or "sample").rstrip("._")

generation_records = _load_generation_summary_for_metric_export()
manifest_by_sample_id = _load_manifest_records_by_sample_id(RUN_MANIFEST)

metric_records = []
missing_generated = []

for row_position, gen in enumerate(generation_records):
    sample_id = gen.get("sample_id")
    manifest_record = manifest_by_sample_id.get(sample_id, {})
    image_id = int(gen.get("image_id", manifest_record.get("image_id", -1)))
    file_name = gen.get("file_name", manifest_record.get("file_name"))
    source_generated_path = Path(gen["generated_path"])

    if not source_generated_path.exists():
        missing_generated.append(str(source_generated_path))
        continue

    metric_index = int(gen.get("index", row_position))
    canonical_name = (
        f"{metric_index:06d}__"
        f"coco_{image_id:012d}__"
        f"{_safe_name(sample_id)}__generated.png"
    )
    metric_generated_path = METRIC_EXPORT_GENERATED_DIR / canonical_name

    if source_generated_path.resolve() != metric_generated_path.resolve():
        shutil.copy2(source_generated_path, metric_generated_path)

    coco_original_path = Path(COCO_ROOT) / "val2017" / file_name if file_name else None
    copied_original_path = None
    if COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT and coco_original_path is not None and coco_original_path.exists():
        original_name = f"{metric_index:06d}__coco_{image_id:012d}__{_safe_name(sample_id)}__original.jpg"
        copied_original_path = METRIC_EXPORT_ORIGINAL_DIR / original_name
        if coco_original_path.resolve() != copied_original_path.resolve():
            shutil.copy2(coco_original_path, copied_original_path)

    metric_record = {
        "metric_index": metric_index,
        "experiment_id": METRIC_EXPORT_EXPERIMENT_ID,
        "method_id": METHOD_ID,
        "sample_id": sample_id,
        "image_id": image_id,
        "file_name": file_name,
        "generated_image_path": str(metric_generated_path),
        "generated_image_relative_path": str(metric_generated_path.relative_to(METRIC_EXPORT_DIR)),
        "source_generated_path": str(source_generated_path),
        "coco_original_path": str(coco_original_path) if coco_original_path is not None else None,
        "copied_original_path": str(copied_original_path) if copied_original_path is not None else None,
        "source_manifest_path": str(RUN_MANIFEST),
        "source_output_dir": str(OUTPUT_DIR) if "OUTPUT_DIR" in globals() else None,
        "background_prompt": manifest_record.get("caption"),
        "foreground_prompts": manifest_record.get("foreground_prompts"),
        "category_names": manifest_record.get("category_names"),
        "category_ids": manifest_record.get("category_ids"),
        "annotation_ids": manifest_record.get("annotation_ids"),
        "area_ratios": manifest_record.get("area_ratios"),
        "target_size": manifest_record.get("target_size"),
        "original_size": manifest_record.get("original_size"),
        "model_family": gen.get("model_family", manifest_record.get("model_family")),
        "sampler": gen.get("sampler", "LCMScheduler"),
        "seed": gen.get("seed"),
        "elapsed_sec": gen.get("elapsed_sec"),
        "generation_metadata": gen,
    }
    metric_records.append(metric_record)

if missing_generated:
    raise FileNotFoundError(
        "Một số ảnh generated_path trong summary không tồn tại. Ví dụ: "
        + "; ".join(missing_generated[:5])
    )

with METRIC_EXPORT_MANIFEST_JSONL.open("w", encoding="utf-8") as f:
    for record in metric_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

csv_fields = [
    "metric_index",
    "experiment_id",
    "method_id",
    "sample_id",
    "image_id",
    "file_name",
    "generated_image_path",
    "generated_image_relative_path",
    "source_generated_path",
    "coco_original_path",
    "copied_original_path",
    "background_prompt",
    "foreground_prompts",
    "category_names",
    "annotation_ids",
    "model_family",
    "sampler",
    "seed",
    "elapsed_sec",
]
with METRIC_EXPORT_MANIFEST_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for record in metric_records:
        writer.writerow({
            key: json.dumps(record.get(key), ensure_ascii=False)
            if isinstance(record.get(key), (list, dict))
            else record.get(key)
            for key in csv_fields
        })

export_summary = {
    "experiment_id": METRIC_EXPORT_EXPERIMENT_ID,
    "method_id": METHOD_ID,
    "num_generated_images": len(metric_records),
    "export_dir": str(METRIC_EXPORT_DIR),
    "generated_images_dir": str(METRIC_EXPORT_GENERATED_DIR),
    "manifest_jsonl": str(METRIC_EXPORT_MANIFEST_JSONL),
    "manifest_csv": str(METRIC_EXPORT_MANIFEST_CSV),
    "copy_original_images": COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT,
    "source_manifest_path": str(RUN_MANIFEST),
    "source_output_dir": str(OUTPUT_DIR) if "OUTPUT_DIR" in globals() else None,
    "zip_path": str(METRIC_EXPORT_ZIP_PATH),
}
with METRIC_EXPORT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(export_summary, f, ensure_ascii=False, indent=2)

if METRIC_EXPORT_ZIP_PATH.exists():
    METRIC_EXPORT_ZIP_PATH.unlink()
shutil.make_archive(
    str(METRIC_EXPORT_ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=METRIC_EXPORT_DIR,
)
export_summary["zip_size_mb"] = round(METRIC_EXPORT_ZIP_PATH.stat().st_size / (1024 * 1024), 2)
with METRIC_EXPORT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(export_summary, f, ensure_ascii=False, indent=2)

display(Markdown(
    "## Metric Export Ready\n"
    f"- Experiment: `{METRIC_EXPORT_EXPERIMENT_ID}`\n"
    f"- Method: `{METHOD_ID}`\n"
    f"- Generated images: `{len(metric_records)}`\n"
    f"- Folder ảnh sinh: `{METRIC_EXPORT_GENERATED_DIR}`\n"
    f"- Manifest JSONL: `{METRIC_EXPORT_MANIFEST_JSONL}`\n"
    f"- Manifest CSV: `{METRIC_EXPORT_MANIFEST_CSV}`\n"
    f"- Zip tải về: `{METRIC_EXPORT_ZIP_PATH}`\n"
    f"- Zip size: `{export_summary.get('zip_size_mb')} MB`"
))

if "pd" in globals():
    display(pd.DataFrame(metric_records)[[
        "metric_index",
        "sample_id",
        "image_id",
        "file_name",
        "generated_image_relative_path",
        "coco_original_path",
        "background_prompt",
    ]].head())
else:
    print(json.dumps(export_summary, ensure_ascii=False, indent=2))


## 2. Đo metric sau generation

Phần dưới dùng ảnh đã sinh trong `OUTPUT_DIR` và `generation_summary.json` để tính:
`FID`, `IS`, `CLIP(fg)`, `CLIP(bg)`, `Time(s)`

Trước khi đo metric, notebook giải phóng pipeline diffusion khỏi GPU để có chỗ load Inception/CLIP.


In [ ]:
# Giải phóng VRAM trước khi load Inception/CLIP cho metric.
import gc

for var_name in ("smd", "runtime", "capture", "generated", "payload", "overlay", "original", "batch", "loader", "preview_batch"):
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("[OK] Released generation objects before metric evaluation.")


In [ ]:
# Đo FID, IS, CLIP(fg), CLIP(bg), Time(s) trên full output vừa generate.
from metrics import MetricEvaluationConfig, run_evaluation, write_metrics_report
import pandas as pd
import math

generation_summary_path = OUTPUT_DIR / "generation_summary.json"
assert generation_summary_path.exists(), f"Missing generation summary: {generation_summary_path}"

metric_device = "cuda:0" if torch.cuda.is_available() else "cpu"
metric_config = MetricEvaluationConfig(
    manifest_path=RUN_MANIFEST,
    coco_root=COCO_ROOT,
    generated_dir=OUTPUT_DIR,
    generation_summary=generation_summary_path,
    output_dir=METRICS_OUTPUT_DIR,
    model_family="sd15",
    target_size=TARGET_SIZE,
    metrics=METRIC_NAMES,
    batch_size=METRIC_BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    device=metric_device,
    clip_batch_size=CLIP_BATCH_SIZE,
    is_splits=IS_SPLITS,
)

metric_report = run_evaluation(metric_config)
metrics_json, metrics_csv = write_metrics_report(
    metric_report,
    METRICS_OUTPUT_DIR,
    prefix=METRICS_REPORT_PREFIX,
)

values = metric_report["metrics"]

def fmt(value: object, digits: int = 4) -> str:
    if value is None:
        return "-"
    try:
        value = float(value)
        if math.isnan(value):
            return "-"
        return f"{value:.{digits}f}"
    except Exception:
        return str(value)

metrics_table = pd.DataFrame([
    {"Metric": "Method", "Value": METHOD_ID},
    {"Metric": "FID↓", "Value": fmt(values.get("fid"))},
    {"Metric": "IS↑", "Value": fmt(values.get("is_mean"))},
    {"Metric": "IS std", "Value": fmt(values.get("is_std"))},
    {"Metric": "CLIP(fg)↑", "Value": fmt(values.get("clip_fg_x100"))},
    {"Metric": "CLIP(bg)↑", "Value": fmt(values.get("clip_bg_x100"))},
    {"Metric": "Time(s)↓", "Value": fmt(values.get("time_mean_sec"))},
    {"Metric": "Total time(s)", "Value": fmt(values.get("time_total_sec"))},
])

display(Markdown(
    f"## Metric Evaluation Done ({METHOD_ID})\n"
    f"- evaluated: `{metric_report['num_evaluated']}` / `{metric_report['num_manifest_records']}` samples\n"
    f"- missing generated images: `{metric_report['num_missing_generated']}`\n"
    f"- metrics JSON: `{metrics_json}`\n"
    f"- metrics CSV: `{metrics_csv}`"
))
display(metrics_table)

metric_report
